# PCT Tuning and Eval

Uses checkpoints to fine-tune PCT models with the other dataset (full, partial) that they were originally trained on.

## Prep

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
!git clone --recursive --single-branch --branch PCT-retrain https://github.com/DavidClaszen/pointcloud-bench /content/pointcloud-bench
%cd /content/pointcloud-bench
%pip install -r envs/pct/requirements.txt

In [ ]:
# Paths, folders
import os

REPO_PATH = '/content/pointcloud-bench'
DRIVE_PATH = '/content/drive/MyDrive/pointcloud-bench'
results_dir = os.path.join(DRIVE_PATH, 'results')
os.makedirs(results_dir, exist_ok=True)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import accuracy_score, confusion_matrix
from hydra import initialize_config_dir, compose
from omegaconf import OmegaConf
from IPython.display import display, HTML
from tqdm import tqdm

In [ ]:
# Extreme partiality is another option for tuning, uncomment if required
# Copy select files from Google Drive to the repo dataset folder
!rsync -avP {DRIVE_PATH}/datasets/fullmodelnet40.tar.gz {REPO_PATH}/datasets
!rsync -avP {DRIVE_PATH}/datasets/partialmodelnet40.tar.gz {REPO_PATH}/datasets
# !rsync -avP {DRIVE_PATH}/datasets/pm40_extreme_partiality.tar.gz {REPO_PATH}/datasets

# Extract zip files
%cd {REPO_PATH}
!tar -xvzf datasets/fullmodelnet40.tar.gz -C datasets
!tar -xvzf datasets/partialmodelnet40.tar.gz -C datasets
# !tar -xvzf datasets/pm40_extreme_partiality.tar.gz -C datasets

In [ ]:
# Original data used by Point-Transformers, from ShapeNet
# Kept for archival reasons; uncomment if required later
# ZIP = '/content/drive/MyDrive/pointcloud-bench/datasets/modelnet40_normal_resampled.zip'
# DEST = '/content/pointcloud-bench/datasets/modelnet40_pct'
# !unzip -q -n '{ZIP}' -d '{DEST}'
# !find '{DEST}' -type f | wc -l

## Checkpoints

You can get the trained model checkpoints from [this folder](https://drive.google.com/drive/folders/13TIEUMSkxi-MxY_Y-kTkAsEcRP5OgAR3?usp=sharing).

Copy those files into your own Google Drive and, if necessary, change the first path in the code below. Contents:

- Main models:
    - pct_p50.pth:      PCT trained on partial50 from PAPNet
    - pct_pfull.pth:    PCT trained on full version of PAPNet
- Archived:
    - pct_pct.pth:      PCT trained on the original dataset used for PCT
    - pct_pct_p50:      PCT trained on the original dataset used for PCT, tuned on partial50 from PAPNet
    - pct_p50_pct.pth:  PCT trained on partial 50 from PAPNet, tuned on original dataset used for PCT

In [ ]:
!rsync -avP {DRIVE_PATH}/checkpoints/. {REPO_PATH}/checkpoints

## Tuning


In [ ]:
%cd /content/pointcloud-bench/repos/Point-Transformers
!python train_cls.py --help

In [ ]:
# Tune PCT model trained on full data further on partial
!python train_cls.py model=Menghao use_papnet_loader=True batch_size=256 learning_rate=0.0005 epoch=225 workers=4 data_path=../../datasets/modelnet40_partial/ checkpoint_path=../../../../../checkpoints/pct_full.pth step_size=10

In [ ]:
# Zip Point-Transformers logs
!zip -r results.zip ./log/cls/Menghao/

## Eval

In [ ]:
# Function for inference

from test_cls import get_predictions

%cd {REPO_PATH}/repos/Point-Transformers/
config_dir = f'{REPO_PATH}/repos/Point-Transformers/config'


def test_pct(
    data_folder: str = 'partialmodelnet40',
    checkpoint: str = 'pct_p50',
    config_dir: str = config_dir,
    partiality: str = ''
) -> tuple[np.array, np.array]:
    """Gets predictions from pretrained PCT model.

    Args:
        data_folder (str, optional): Dataset folder to test on.
            Defaults to 'partialmodelnet40'.
        checkpoint (str, optional): Checkpoint name; omit extension.
            Defaults to 'pct_p50'.
        config_dir (str, optional): Where the model configs reside.
            Only necessary to run, but not used. Defaults to config_dir.
        partiality (str, optional): Used only for extreme partiality
            datasets with non-standard filenames. Use '_10', '_20',
            '_30', '_40'.

    Returns:
        tuple(np.array, np.array): Tuple of predictions and base truths
    """
    if data_folder in ['partialmodelnet40', 'pm40_partiality_level', 'fullmodelnet40']:
        papnet_loader = 'true'
    else: papnet_loader = 'false'

    with initialize_config_dir(config_dir=config_dir, version_base='1.2'):
        cfg = compose(
            config_name='cls',
            overrides=[
                f'data_path=../../datasets/{data_folder}/',
                f'checkpoint_path=../../checkpoints/{checkpoint}.pth',
                f'use_papnet_loader={papnet_loader}',
                f'partiality={partiality}'
            ],
        )
    OmegaConf.set_struct(cfg, False)
    y_true, y_pred = get_predictions(cfg)
    accuracy = accuracy_score(y_true, y_pred)
    print(f'Accuracy: {accuracy:.4f} for dataset {data_folder} and model {checkpoint}')
    return (y_true, y_pred)

In [ ]:
# Runs everything as a normal function instead
# Train PCT, test PCT


config_dir = "/content/pointcloud-bench/repos/Point-Transformers/config"

with initialize_config_dir(config_dir=config_dir, version_base="1.2"):
    cfg = compose(
        config_name="cls",
        overrides=[
            "data_path=../../datasets/modelnet40_partial/",
            "checkpoint_path=../../checkpoints/pct_full.pth",
            "use_papnet_loader=true"
        ],
    )

OmegaConf.set_struct(cfg, False)
y_true_00, y_pred_00 = get_predictions(cfg)
accuracy_score(y_pred_00, y_true_00)

In [ ]:
df = pd.DataFrame({
    'y_true':      y_true_00,
    'pct_pretrain_pap_tune':     y_pred_00
})

df.to_csv('pct_results.csv', index=False, sep=';')